# Master Guide: Decoding Ranking Problems (Third Maximum Number)

## 1. How to Think About the Problem: The Mental Model

When faced with a problem asking for the "Third Maximum" or "Kth Largest" element, your brain's first instinct is usually to sort the array.

* **The Naive Thought:** Sort the array in descending order, remove duplicates, and pick the element at index 2.
* **The Catch:** Sorting takes $O(n \log n)$ time. An interviewer will usually ask: *"Can you do this in a single pass ($O(n)$ time)?"*
* **The Single-Pass Strategy (The Podium Metaphor):** Imagine a podium with three spots: Gold (`max1`), Silver (`max2`), and Bronze (`max3`). As you walk through the array, incoming numbers challenge the champions. If a number is bigger than Gold, Gold gets pushed down to Silver, Silver gets pushed down to Bronze, and the new number takes Gold.

## 2. Approach 1: Primitive State Tracking (The Hard Way)

This is where most developers start—using primitive variables and boolean flags to track whether slots are legitimately filled.

### The Pitfalls & Edge Cases

1. **The `Integer.MIN_VALUE` Trap:** If your array contains $-2,148,483,648$, and you initialized your variables to `0` or `Integer.MIN_VALUE`, your code will mistake a valid negative number for an "empty" slot.
* *Fix:* Use `long` data types and initialize variables to `Long.MIN_VALUE` to create a safe mathematical buffer.

<font color = #FF0000>NOTE : </font> **Type Boundaries** - Always match your type boundaries (Long.MIN_VALUE with long, Integer.MIN_VALUE with int).


2. **The Duplicate Trap:** If the array contains `[2, 2, 3]`, a naive shift will fill `max1 = 3`, `max2 = 2`, and `max3 = 2`. That's two champions with the same value!
* *Fix:* Strict inequality guards (`nums[i] != max1`, `nums[i] != max2`) are mandatory at every step.


3. **The Cascade (Manual) Flag Trap:** When champions get shoved down from Gold to Silver to Bronze, boolean validation flags (`hasMax2`, `hasMax3`) must be explicitly flipped *only* if the underlying value isn't a sentinel placeholder. Missing this causes valid runner-ups to be ignored.

### Refined Working Code (Primitives)

In [3]:
class Solution1 {
    public static int thirdMax(int[] nums) {
        long max1 = Long.MIN_VALUE;
        long max2 = Long.MIN_VALUE;
        long max3 = Long.MIN_VALUE;
        boolean hasMax2 = false;
        boolean hasMax3 = false;

        for (int i = 0; i < nums.length; i++) {
            if (nums[i] > max1) {
                max3 = max2;
                max2 = max1;
                max1 = nums[i];
                if (max2 != Long.MIN_VALUE) hasMax2 = true;
                if (max3 != Long.MIN_VALUE) hasMax3 = true;
            } else if ((nums[i] > max2) && (nums[i] != max1)) {
                max3 = max2;
                max2 = nums[i];
                hasMax2 = true;
                if (max3 != Long.MIN_VALUE) hasMax3 = true;
            } else if ((nums[i] > max3) && hasMax2 && (nums[i] != max2) && (nums[i] != max1)) {
                max3 = nums[i];
                hasMax3 = true;
            }
        }
        return hasMax3 ? (int)max3 : (int)max1;
    }
}

## 3. Approach 2: The Clean Standard (`Integer` & `null` References)

To bypass the nightmare of boolean flags, sentinel values, and missed cascade updates, use wrapper objects. `null` naturally represents an unassigned podium spot.

## What continue Does in Java
In a loop (for, while), the continue keyword instantly stops the current iteration right where it is. It skips all remaining lines of code inside the loop body for that specific element and jumps straight to the next loop step (incrementing i).

In this algorithm, continue acts as an upfront bouncer/filter:

If an incoming number already exists in your podium (m1, m2, or m3), the code hits continue.

This instantly throws the duplicate away and moves to the next number in the array, completely preventing duplicate values from entering the ranking logic.

In [4]:
class Solution2 {
    public static int thirdMax(int[] nums) {
        Integer m1 = null;
        Integer m2 = null;
        Integer m3 = null;
        
        for (int i = 0; i < nums.length; i++) {
            // Upfront duplicate filter
            if ((m1 != null && nums[i] == m1) || (m2 != null && nums[i] == m2) || (m3 != null && nums[i] == m3)) {
                continue;
            }
            
            if (m1 == null || nums[i] > m1) {
                m3 = m2;
                m2 = m1;
                m1 = nums[i];
            } else if (m2 == null || nums[i] > m2) {
                m3 = m2;
                m2 = nums[i];
            } else if (m3 == null || nums[i] > m3) {
                m3 = nums[i];
            }
        }
        
        return m3 == null ? m1 : m3;
    }
}

<font color = red>Note - Primitive state tracking is so fragile </font>

# Test Cases

In [9]:
public class Main{
    public static void main(String[] args) {
        
            System.out.println("=== Running Third Max Tests ===" );
            int[][] testCases3rd = {
                {3, 2, 1}, {1, 2}, {2, 2, 3, 1}, 
                {3, -2, 1, -2}, {1, 2, 2, 5, 3, 5}, {-1, -2, -3, -4}
            };
            for (int[] tc : testCases3rd) {
                System.out.println("Input: " + Arrays.toString(tc) + " => Output: " + Solution1.thirdMax(tc));
                System.out.println("Input: " + Arrays.toString(tc) + " => Output: " + Solution2.thirdMax(tc));
                System.out.println("====");
            }
    }
}

In [10]:
Main.main(new String[]{});

=== Running Third Max Tests ===
Input: [3, 2, 1] => Output: 1
Input: [3, 2, 1] => Output: 1
====
Input: [1, 2] => Output: 2
Input: [1, 2] => Output: 2
====
Input: [2, 2, 3, 1] => Output: 1
Input: [2, 2, 3, 1] => Output: 1
====
Input: [3, -2, 1, -2] => Output: -2
Input: [3, -2, 1, -2] => Output: -2
====
Input: [1, 2, 2, 5, 3, 5] => Output: 2
Input: [1, 2, 2, 5, 3, 5] => Output: 2
====
Input: [-1, -2, -3, -4] => Output: -3
Input: [-1, -2, -3, -4] => Output: -3
====


# Other Links

https://leetcode.com/problems/third-maximum-number/solutions/2614958/third-maximum-number